In [5]:
import os
import sys
import pandas as pd
import numpy as np
from sklearn.ensemble import IsolationForest
import joblib
from pathlib import Path
from dotenv import load_dotenv

sys.path.insert(0, os.path.abspath('../src'))
from db.connection import read_sql
from features.system_features import extract_system_features
from features.llm_features import extract_llm_features

load_dotenv()
print("✓ Imports OK")

✓ Imports OK


In [6]:
# Load data from SAP HANA
df_sys_peak = read_sql("""
SELECT TIMESTAMP, LOG_TYPE, HTTP_STATUS_CODE, CLIENT_IP, SERVICE_ID
FROM SAP_SYSTEM_LOGS
WHERE EXTRACT(HOUR FROM TIMESTAMP) BETWEEN 8 AND 18
    AND DAYOFWEEK(TIMESTAMP) BETWEEN 2 AND 6
LIMIT 100000
""")

df_sys_offpeak = read_sql("""
SELECT TIMESTAMP, LOG_TYPE, HTTP_STATUS_CODE, CLIENT_IP, SERVICE_ID
FROM SAP_SYSTEM_LOGS
WHERE (EXTRACT(HOUR FROM TIMESTAMP) < 8 OR EXTRACT(HOUR FROM TIMESTAMP) > 18)
    OR DAYOFWEEK(TIMESTAMP) NOT BETWEEN 2 AND 6
LIMIT 100000
""")

df_llm_peak = read_sql("""
SELECT TIMESTAMP, LOG_TYPE, LLM_MODEL_ID, LLM_STATUS, LLM_COST_USD, LLM_RESPONSE_TIME_MS
FROM SAP_LLM_LOGS
WHERE EXTRACT(HOUR FROM TIMESTAMP) BETWEEN 8 AND 18
    AND DAYOFWEEK(TIMESTAMP) BETWEEN 2 AND 6
LIMIT 100000
""")

df_llm_offpeak = read_sql("""
SELECT TIMESTAMP, LOG_TYPE, LLM_MODEL_ID, LLM_STATUS, LLM_COST_USD, LLM_RESPONSE_TIME_MS
FROM SAP_LLM_LOGS
WHERE (EXTRACT(HOUR FROM TIMESTAMP) < 8 OR EXTRACT(HOUR FROM TIMESTAMP) > 18)
    OR DAYOFWEEK(TIMESTAMP) NOT BETWEEN 2 AND 6
LIMIT 100000
""")

print(f"System PEAK:    {len(df_sys_peak)} records")
print(f"System OFFPEAK: {len(df_sys_offpeak)} records")
print(f"LLM PEAK:       {len(df_llm_peak)} records")
print(f"LLM OFFPEAK:    {len(df_llm_offpeak)} records")
print("✓ Data loaded")

d:\DocumentsD\CodigosD\MLpractica\SAP---AI-Security-Anomaly-Detection\sap-soc-hackathon\backend\src\db\connection.py:56: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(sql, conn)


System PEAK:    100000 records
System OFFPEAK: 100000 records
LLM PEAK:       100000 records
LLM OFFPEAK:    100000 records
✓ Data loaded


In [7]:
print("Extracting System PEAK features...")
sys_peak_features = extract_system_features(df_sys_peak)
print(f"System PEAK features: {sys_peak_features.shape}")
print(sys_peak_features.head())

Extracting System PEAK features...
System PEAK features: (100000, 9)
   LOG_TYPE_ENCODED  HTTP_STATUS_NORM  CLIENT_IP_ENCODED  SERVICE_ID_ENCODED  \
0                 0               0.0           0.000000                 0.0   
1                 0               0.0           0.009615                 0.0   
2                 0               0.0           0.019231                 0.0   
3                 0               0.0           0.000000                 0.0   
4                 0               0.0           0.028846                 0.0   

   IS_PEAK  IS_WEEKEND  HOUR  IP_REQ_COUNT_5M  IP_ERROR_RATIO  
0        1       False    13         0.000923               0  
1        1       False    13         0.000923               0  
2        1       False    13         0.000923               0  
3        1       False    13         0.001847               0  
4        1       False    13         0.000923               0  


In [8]:
print("Extracting System OFFPEAK features...")
sys_offpeak_features = extract_system_features(df_sys_offpeak)
print(f"System OFFPEAK features: {sys_offpeak_features.shape}")
print(sys_offpeak_features.head())

Extracting System OFFPEAK features...
System OFFPEAK features: (100000, 9)
   LOG_TYPE_ENCODED  HTTP_STATUS_NORM  CLIENT_IP_ENCODED  SERVICE_ID_ENCODED  \
0                 0               0.0           0.000000                 0.0   
1                 0               0.0           0.009615                 0.0   
2                 0               0.0           0.019231                 0.0   
3                 0               0.0           0.028846                 0.0   
4                 0               0.0           0.038462                 0.0   

   IS_PEAK  IS_WEEKEND  HOUR  IP_REQ_COUNT_5M  IP_ERROR_RATIO  
0        0       False     0          0.00088               0  
1        0       False     0          0.00088               0  
2        0       False     0          0.00088               0  
3        0       False     0          0.00088               0  
4        0       False     0          0.00088               0  


In [9]:
print("Extracting LLM PEAK features...")
llm_peak_features = extract_llm_features(df_llm_peak)
print(f"LLM PEAK features: {llm_peak_features.shape}")
print(llm_peak_features.head())

Extracting LLM PEAK features...
LLM PEAK features: (100000, 10)
   LOG_TYPE_ENCODED  LLM_STATUS_ENCODED  LLM_COST_NORM  \
0                 0                 0.0       0.070555   
1                 0                 0.0       0.044579   
2                 0                 0.0       0.057539   
3                 0                 0.0       0.066695   
4                 0                 0.0       0.065460   

   LLM_RESPONSE_TIME_NORM  LLM_MODEL_ENCODED  IS_PEAK  IS_WEEKEND  HOUR  \
0                0.028622                0.0        1       False    11   
1                0.022242                0.0        1       False    11   
2                0.402713                0.0        1       False    11   
3                0.079974                0.0        1       False    11   
4                0.079399                0.0        1       False    11   

   MODEL_REQ_COUNT  MODEL_ERROR_RATE  
0         0.000096                 0  
1         0.000193                 0  
2         0.000289 

In [10]:
print("Extracting LLM OFFPEAK features...")
llm_offpeak_features = extract_llm_features(df_llm_offpeak)
print(f"LLM OFFPEAK features: {llm_offpeak_features.shape}")
print(llm_offpeak_features.head())

Extracting LLM OFFPEAK features...
LLM OFFPEAK features: (100000, 10)
   LOG_TYPE_ENCODED  LLM_STATUS_ENCODED  LLM_COST_NORM  \
0                 0                 0.0       0.058390   
1                 0                 0.0       0.056031   
2                 0                 0.0       0.065382   
3                 0                 0.0       0.073295   
4                 0                 0.0       0.080467   

   LLM_RESPONSE_TIME_NORM  LLM_MODEL_ENCODED  IS_PEAK  IS_WEEKEND  HOUR  \
0                0.308543                0.0        0       False     0   
1                0.383948                0.0        0       False     0   
2                0.027759                0.0        0       False     0   
3                0.194776                0.0        0       False     0   
4                0.238915                0.0        0       False     0   

   MODEL_REQ_COUNT  MODEL_ERROR_RATE  
0         0.000073                 0  
1         0.000146                 0  
2         0.0

In [11]:
def train_isolation_forest(features_df, model_name, contamination=0.02, n_estimators=100):
    """Entrena modelo IF y lo guarda en artifacts/"""
    print(f"\nTraining {model_name}...")
    
    features_df = features_df.dropna()
    
    if len(features_df) < 100:
        raise ValueError(f"Not enough data: {len(features_df)} rows")
    
    model = IsolationForest(
        contamination=contamination,
        n_estimators=n_estimators,
        random_state=42,
        n_jobs=-1
    )
    
    predictions = model.fit_predict(features_df)
    scores = model.score_samples(features_df)
    
    # Guardar modelo
    artifact_path = Path('../../artifacts') / f'{model_name}.joblib'
    artifact_path.parent.mkdir(parents=True, exist_ok=True)
    joblib.dump(model, artifact_path)
    print(f"✓ Saved to {artifact_path}")
    
    # Stats
    print(f"  Training records: {len(features_df)}")
    print(f"  Anomalies detected: {(predictions == -1).sum()}")
    print(f"  Score range: [{scores.min():.3f}, {scores.max():.3f}]")
    print(f"  Score P5: {np.percentile(scores, 5):.3f}")
    print(f"  Score P95: {np.percentile(scores, 95):.3f}")
    
    return model, scores

print("✓ Training function defined")

✓ Training function defined


In [12]:
model_sys_peak, scores_sys_peak = train_isolation_forest(
    sys_peak_features, 
    'IF_SYSTEM_PEAK', 
    contamination=0.02
)


Training IF_SYSTEM_PEAK...
✓ Saved to ..\..\artifacts\IF_SYSTEM_PEAK.joblib
  Training records: 100000
  Anomalies detected: 2000
  Score range: [-0.646, -0.452]
  Score P5: -0.583
  Score P95: -0.474


In [13]:
model_sys_offpeak, scores_sys_offpeak = train_isolation_forest(
    sys_offpeak_features, 
    'IF_SYSTEM_OFFPEAK', 
    contamination=0.04
)


Training IF_SYSTEM_OFFPEAK...
✓ Saved to ..\..\artifacts\IF_SYSTEM_OFFPEAK.joblib
  Training records: 100000
  Anomalies detected: 4000
  Score range: [-0.696, -0.393]
  Score P5: -0.606
  Score P95: -0.411


In [14]:
model_llm_peak, scores_llm_peak = train_isolation_forest(
    llm_peak_features, 
    'IF_LLM_PEAK', 
    contamination=0.02
)


Training IF_LLM_PEAK...
✓ Saved to ..\..\artifacts\IF_LLM_PEAK.joblib
  Training records: 100000
  Anomalies detected: 2000
  Score range: [-0.742, -0.410]
  Score P5: -0.599
  Score P95: -0.431


In [15]:
model_llm_offpeak, scores_llm_offpeak = train_isolation_forest(
    llm_offpeak_features, 
    'IF_LLM_OFFPEAK', 
    contamination=0.04
)


Training IF_LLM_OFFPEAK...
✓ Saved to ..\..\artifacts\IF_LLM_OFFPEAK.joblib
  Training records: 100000
  Anomalies detected: 4000
  Score range: [-0.732, -0.412]
  Score P5: -0.609
  Score P95: -0.432


In [16]:
artifacts_dir = Path('../../artifacts')
models = list(artifacts_dir.glob('*.joblib'))
print("\n=== Models Saved ===")
for model_file in models:
    size_mb = model_file.stat().st_size / (1024 * 1024)
    print(f"✓ {model_file.name} ({size_mb:.2f} MB)")

print(f"\n✓ PHASE 2B: Training complete - {len(models)} models saved")


=== Models Saved ===
✓ IF_LLM_OFFPEAK.joblib (1.46 MB)
✓ IF_LLM_PEAK.joblib (1.49 MB)
✓ IF_SYSTEM_OFFPEAK.joblib (1.45 MB)
✓ IF_SYSTEM_PEAK.joblib (1.88 MB)

✓ PHASE 2B: Training complete - 4 models saved
